# 03 — YOLOv5 evaluation on its own dataset

In [1]:
from pathlib import Path
import os
import sys

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (CURRENT, *CURRENT.parents) if (path / "src").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root табылмады. Notebook-ты project папкасының ішінен ашыңыз."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.executable)


PROJECT_ROOT: C:\Users\Ansagan\Downloads\smart_vision_final_project_laptop_cpu
Python: c:\Users\Ansagan\venvs\ml-yolo-cpu\Scripts\python.exe


In [2]:
import json

from src.config import METRICS_DIR, YOLOV5_SPEC, yolo5_data_yaml, yolo5_weights
from src.eval_yolo import eval_with_ultralytics, eval_yolov5_repo, save_metrics
from src.yolo_dataset_utils import dataset_diagnostics

DATA_YAML = yolo5_data_yaml()
WEIGHTS = yolo5_weights()
print("YOLOv5 weights:", WEIGHTS)
print("YOLOv5 runtime YAML:", DATA_YAML)
print(dataset_diagnostics(DATA_YAML).to_string(index=False))


c:\Users\Ansagan\venvs\ml-yolo-cpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


YOLOv5 weights: C:\Users\Ansagan\Documents\Project\yolov5\runs\train\exp\weights\best.pt
YOLOv5 runtime YAML: C:\Users\Ansagan\Downloads\smart_vision_final_project_laptop_cpu\outputs\runtime\yolov5\data_runtime.yaml
split                                                                                           resolved_path  path_exists  images  label_files  objects missing_label_examples
train C:\Users\Ansagan\Downloads\smart_vision_final_project_laptop_cpu\outputs\runtime\yolov5\lists\train.txt         True     736          736     1303                       
  val   C:\Users\Ansagan\Downloads\smart_vision_final_project_laptop_cpu\outputs\runtime\yolov5\lists\val.txt         True     184          184      340                       
 test   C:\Users\Ansagan\Downloads\smart_vision_final_project_laptop_cpu\outputs\runtime\yolov5\lists\val.txt         True     184          184      340                       


In [3]:
from pathlib import Path
import sys

import torch
import yaml


YOLOV5_REPO = Path(
    r"C:\Users\Ansagan\Documents\Project\yolov5"
)

YOLOV5_WEIGHTS = Path(
    r"C:\Users\Ansagan\Documents\Project"
    r"\yolov5\runs\train\exp\weights\best.pt"
)

CURRENT_RUNTIME_YAML = Path(DATA_YAML)


# Original YOLOv5 модель кластарын импорттау үшін.
if str(YOLOV5_REPO) not in sys.path:
    sys.path.insert(0, str(YOLOV5_REPO))


# Бұл өзің оқытқан сенімді checkpoint болғандықтан,
# weights_only=False қолданамыз.
checkpoint = torch.load(
    YOLOV5_WEIGHTS,
    map_location="cpu",
    weights_only=False,
)

checkpoint_model = (
    checkpoint.get("ema")
    or checkpoint.get("model")
)

raw_names = checkpoint_model.names

if isinstance(raw_names, dict):
    model_names = [
        str(raw_names[index])
        for index in sorted(
            raw_names,
            key=int,
        )
    ]
else:
    model_names = [
        str(name)
        for name in raw_names
    ]


print("Checkpoint класс саны:", len(model_names))
print("Checkpoint кластары:")

for class_id, class_name in enumerate(model_names):
    print(
        f"{class_id}: {class_name}"
    )


runtime_config = yaml.safe_load(
    CURRENT_RUNTIME_YAML.read_text(
        encoding="utf-8"
    )
) or {}

print(
    "\nЕскі runtime YAML класс саны:",
    runtime_config.get(
        "nc",
        len(
            runtime_config.get(
                "names",
                []
            )
        ),
    ),
)

print(
    "Ескі names:",
    runtime_config.get("names"),
)

C:\Users\Ansagan\Documents\Project\yolov5\utils\general.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg


Checkpoint класс саны: 5
Checkpoint кластары:
0: person
1: velo
2: basty
3: belgisiz
4: car

Ескі runtime YAML класс саны: 8
Ескі names: ['person', 'velo', 'belgisiz', 'car', 'basty', 'belgi1', 'belgi2', 'belgi3']


In [4]:
from collections import Counter
from pathlib import Path


YOLOV5_LABELS_ROOT = Path(
    r"C:\Users\Ansagan\Documents\Project\dataset\labels"
)

if not YOLOV5_LABELS_ROOT.exists():
    raise FileNotFoundError(
        "YOLOv5 labels папкасы табылмады:\n"
        f"{YOLOV5_LABELS_ROOT}"
    )


class_counter = Counter()
invalid_label_lines = []

label_files = sorted(
    YOLOV5_LABELS_ROOT.rglob("*.txt")
)

for label_path in label_files:
    lines = label_path.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines()

    for line_number, line in enumerate(
        lines,
        start=1,
    ):
        parts = line.strip().split()

        if not parts:
            continue

        try:
            class_id = int(
                float(parts[0])
            )

        except ValueError:
            invalid_label_lines.append(
                (
                    str(label_path),
                    line_number,
                    line,
                )
            )
            continue

        class_counter[class_id] += 1


print("Label файлдарының саны:", len(label_files))
print("Dataset class ID:", sorted(class_counter))
print("Әр класс объектілерінің саны:", dict(class_counter))

if invalid_label_lines:
    print(
        "Қате label жолдарының саны:",
        len(invalid_label_lines),
    )


invalid_class_ids = [
    class_id
    for class_id in class_counter
    if class_id >= len(model_names)
]

if invalid_class_ids:
    raise ValueError(
        "Бұл dataset ішінде YOLOv5 моделіне жатпайтын "
        f"class ID бар: {invalid_class_ids}\n"
        f"Модель тек 0–{len(model_names) - 1} "
        "кластарын қолдайды.\n"
        "YOLOv5 оқытылған бастапқы 5-классты "
        "labels папкасын қолдану қажет."
    )

print(
    "\nDataset модельдің 5 класына сәйкес келеді."
)

Label файлдарының саны: 921
Dataset class ID: [0, 1, 2, 3, 4]
Әр класс объектілерінің саны: {2: 181, 3: 197, 4: 872, 0: 217, 1: 181}

Dataset модельдің 5 класына сәйкес келеді.


In [5]:
from pathlib import Path

import yaml


corrected_runtime_config = dict(
    runtime_config
)

corrected_runtime_config["nc"] = len(
    model_names
)

corrected_runtime_config["names"] = (
    model_names
)


CORRECTED_DATA_YAML = (
    CURRENT_RUNTIME_YAML.parent
    / "data_runtime_5classes.yaml"
)

CORRECTED_DATA_YAML.write_text(
    yaml.safe_dump(
        corrected_runtime_config,
        allow_unicode=True,
        sort_keys=False,
    ),
    encoding="utf-8",
)


print(
    "Түзетілген runtime YAML:",
    CORRECTED_DATA_YAML,
)

print(
    CORRECTED_DATA_YAML.read_text(
        encoding="utf-8"
    )
)

DATA_YAML = CORRECTED_DATA_YAML

Түзетілген runtime YAML: C:\Users\Ansagan\Downloads\smart_vision_final_project_laptop_cpu\outputs\runtime\yolov5\data_runtime_5classes.yaml
train: C:/Users/Ansagan/Downloads/smart_vision_final_project_laptop_cpu/outputs/runtime/yolov5/lists/train.txt
val: C:/Users/Ansagan/Downloads/smart_vision_final_project_laptop_cpu/outputs/runtime/yolov5/lists/val.txt
test: C:/Users/Ansagan/Downloads/smart_vision_final_project_laptop_cpu/outputs/runtime/yolov5/lists/val.txt
nc: 5
names:
- person
- velo
- basty
- belgisiz
- car



In [6]:
import json
import os

from src.eval_yolo import (
    eval_yolov5_repo,
    save_metrics,
)

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

metrics_v5 = eval_yolov5_repo(
    yolov5_repo=YOLOV5_SPEC.model_root,
    weights=WEIGHTS,
    data_yaml=DATA_YAML,
    imgsz=640,
    device="cpu",
)

save_metrics(
    metrics_v5,
    METRICS_DIR / "yolov5_metrics.json",
)

print(
    json.dumps(
        metrics_v5,
        indent=2,
        ensure_ascii=False,
    )
)


YOLOv5 validation командасы:
c:\Users\Ansagan\venvs\ml-yolo-cpu\Scripts\python.exe C:\Users\Ansagan\Documents\Project\yolov5\val.py --weights C:\Users\Ansagan\Documents\Project\yolov5\runs\train\exp\weights\best.pt --data C:\Users\Ansagan\Downloads\smart_vision_final_project_laptop_cpu\outputs\runtime\yolov5\data_runtime_5classes.yaml --imgsz 640 --task val --device cpu --batch-size 1 --workers 0 --verbose

C:\Users\Ansagan\Documents\Project\yolov5\utils\general.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg
val: data=C:\Users\Ansagan\Downloads\smart_vision_final_project_laptop_cpu\outputs\runtime\yolov5\data_runtime_5classes.yaml, weights=['C:\\Users\\Ansagan\\Documents\\Project\\yolov5\\runs\\train\\exp\\weights\\best.pt'], batch_size=1, imgsz=640, conf_t